# CoralNet Temporal Analysis

**Two-step workflow:**
1. `preprocess()` — raw CoralNet export → grouped CSV + genera CSV
2. Analysis — load preprocessed CSV → statistics → figures

Replace the paths in the **Configuration** cell below, then run top to bottom.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coralnet_temporal import (
    preprocess,
    build_latlon_template,
    load_coralnet_export,
    compute_community_metrics,
    run_temporal_tests,
    run_genera_analysis,
    prepare_growth_form_data,
    growth_form_stats,
    run_growth_form_tests,
    growth_form_ratio,
    GROWTH_FORM_PRESETS,
)
from coralnet_temporal.temporal import bray_curtis_dissimilarity
from coralnet_temporal.genera import compute_genera_stats
from coralnet_temporal.visualize import (
    plot_community_temporal,
    plot_genera_heatmap,
    plot_top_genera_temporal,
    plot_composition_change,
    plot_transect_temporal,
    plot_growth_form_overview,
    plot_growth_form_by_transect,
    plot_growth_form_temporal,
    plot_growth_form_heatmap,
)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RAW_PATH      = Path('../data/raw/2025_percent_covers.csv')       # raw CoralNet export
LABELSET_PATH = Path('../data/raw/labelset.csv')                  # your labelset
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SITES         = ['PBHR']          # set to None to keep all sites
MONTH_ORDER   = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Nov']  # adjust to your survey
MONTH_A       = 'Jan'             # first time point for Jan–Nov comparisons
MONTH_B       = 'Nov'

## Step 1 — Preprocess

Converts the raw CoralNet export into two clean CSVs:
- **grouped** — one benthic column per functional group (Hard coral, Algae, …)
- **genera** — one column per genus (Acropora, Porites, …)

Lat/lon is **not** added here — paste it into the output CSVs manually after running this cell.

In [ ]:
df_grouped, df_genera = preprocess(
    raw_path=RAW_PATH,
    labelset_path=LABELSET_PATH,
    sites=SITES,
    output_grouped=PROCESSED_DIR / 'percent_covers_grouped.csv',
    output_genera=PROCESSED_DIR  / 'percent_covers_genera.csv',
)

print(f'Grouped : {len(df_grouped)} rows × {df_grouped.shape[1]} columns')
print(f'Genera  : {len(df_genera)} rows × {df_genera.shape[1]} columns')
print(f'Months  : {sorted(df_grouped["month"].unique())}')
print(f'Sites   : {df_grouped["site"].unique().tolist()}')
print()
print('Tip: add lat/lon to the CSVs in data/processed/ before running spatial analyses.')

In [ ]:
# Optional: generate a lat/lon template to fill in
tmpl = build_latlon_template(RAW_PATH, out_path='../data/raw/latlon_lookup_template.csv')
print(f'Template written — fill in lat/lon for {len(tmpl)} site × transect × meter combinations')
tmpl.head()

## Step 2 — Load preprocessed data for analysis

In [ ]:
# Load the grouped CSV (used for community-level analysis)
df, coral_cols, _ = load_coralnet_export(
    PROCESSED_DIR / 'percent_covers_grouped.csv',
    non_coral=['Other Invertebrates', 'Soft Substrate', 'Hard Substrate', 'Other', 'Algae'],
)

print(f'Observations : {len(df)}')
print(f'Unique locations : {df["location_id"].nunique()}')
print(f'Time points : {df["month"].nunique()}')
df.head(3)

## Step 3 — Community metrics

In [ ]:
df = compute_community_metrics(df, coral_cols)
fig = plot_community_temporal(df, metrics=['Total_Coral', 'Richness'], month_order=MONTH_ORDER)
fig.savefig(PROCESSED_DIR / 'community_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Statistical tests

In [ ]:
results = run_temporal_tests(df, metric='Total_Coral', month_order=MONTH_ORDER)

print('Friedman test:', results['friedman'])
print()
print('Pairwise Wilcoxon:')
display(results['pairwise'])
if results['lmm']:
    print('\nVariance partitioning (LMM):', results['lmm'])
print('\nPower analysis:', results['power'])

## Step 5 — Genus-level analysis

In [ ]:
# Load genus-level data
df_gen, gen_cols, _ = load_coralnet_export(
    PROCESSED_DIR / 'percent_covers_genera.csv',
    non_coral=['Sponge','Heliopora','Other sessile','Soft coral',
               'Sand','Rock','Other','Rubble','Coralline algae','Macroalgae','Turf algae'],
)
df_gen = compute_community_metrics(df_gen, gen_cols)

temporal_df, prevalence_df = run_genera_analysis(
    df_gen, gen_cols, month_a=MONTH_A, month_b=MONTH_B
)

sig = temporal_df[temporal_df['p_value'] < 0.05]
print(f'Tested {len(temporal_df)} genera — {len(sig)} significant (p < 0.05)')
display(temporal_df)
display(prevalence_df)

In [ ]:
top5 = compute_genera_stats(df_gen, gen_cols).nlargest(5, 'Mean_Cover (%)').index.tolist()

fig = plot_top_genera_temporal(df_gen, genera=top5, month_order=MONTH_ORDER)
fig.savefig(PROCESSED_DIR / 'top_genera_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_genera_heatmap(df_gen, genera=top5, month_order=MONTH_ORDER)
plt.show()

## Step 6 — Compositional change

In [ ]:
bc = bray_curtis_dissimilarity(df_gen, gen_cols, MONTH_A, MONTH_B)
label = 'VERY STABLE' if bc < 0.1 else ('MINOR CHANGES' if bc < 0.3 else 'SUBSTANTIAL CHANGES')
print(f'Bray-Curtis ({MONTH_A} vs {MONTH_B}): {bc:.4f}  → {label}')

fig = plot_composition_change(df_gen, gen_cols, top5, month_a=MONTH_A, month_b=MONTH_B)
fig.savefig(PROCESSED_DIR / 'composition_change.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Transect patterns (descriptive only)

> n = 4 transects — no spatial inference, descriptive only.

In [ ]:
fig = plot_transect_temporal(df, metric='Hard coral', month_order=MONTH_ORDER)
plt.show()

## Step 8 — Growth form analysis

Drill into any genus that has multiple morphological codes in your labelset.
Change `FOCUS_GENUS` to any key in `GROWTH_FORM_PRESETS`, or pass your own
`form_map` dict for full control.

**Built-in presets:** Porites, Acropora, Echinopora, Montipora, Galaxea,
Hydnopora, Isopora, Merulina

In [ ]:
# ── configuration ─────────────────────────────────────────────────────────────
FOCUS_GENUS = 'Porites'   # change to any preset, e.g. 'Acropora'

# To use a custom genus not in the presets, define your own form_map:
# form_map = {
#     'Branching': ['Ech.branch'],
#     'Encrusting': ['Ech.encrus'],
#     'Foliose': ['Ech.folio'],
# }
# FOCUS_GENUS = 'Echinopora'

form_map  = GROWTH_FORM_PRESETS[FOCUS_GENUS]
form_cols = list(form_map.keys())
print(f'Analysing {FOCUS_GENUS}: {form_cols}')

In [ ]:
# Load and prepare growth form data from the raw CSV
df_gf = prepare_growth_form_data(
    RAW_PATH,
    form_map=form_map,
    sites=SITES,
    genus_name=FOCUS_GENUS,
)
print(f'{len(df_gf)} observations, {df_gf["location_id"].nunique()} locations')
df_gf.head(3)

In [ ]:
# Summary statistics
stats_result = growth_form_stats(df_gf, form_cols, genus_total_col=f'{FOCUS_GENUS}_Total',
                                  month_order=MONTH_ORDER)

print('=== Overall composition ===')
display(stats_result['overall'])

print('\n=== By transect ===')
display(stats_result['by_transect'])

print('\n=== By month ===')
display(stats_result['by_month'])

if 'jan_nov_change' in stats_result:
    print(f'\n=== Jan → Nov change ===')
    display(stats_result['jan_nov_change'])

In [ ]:
# Statistical tests — Friedman repeated-measures per form
tests_df = run_growth_form_tests(df_gf, form_cols)
print('=== Friedman tests by growth form ===')
display(tests_df)

In [ ]:
# Optional: branching:massive (or any pair) ratio
# Change the pair to match your genus's forms
if len(form_cols) >= 2:
    ratio_pair = (form_cols[0], form_cols[-1])  # e.g. Branching:Massive for Porites
    print(f'Ratio {ratio_pair[0]}:{ratio_pair[1]} by transect:')
    display(growth_form_ratio(df_gf, *ratio_pair, by='transect').to_frame())
    print(f'\nRatio {ratio_pair[0]}:{ratio_pair[1]} over time:')
    display(growth_form_ratio(df_gf, *ratio_pair, by='month', month_order=MONTH_ORDER).to_frame())
else:
    ratio_pair = None

In [ ]:
# Figures
fig = plot_growth_form_overview(df_gf, form_cols, genus_name=FOCUS_GENUS)
fig.savefig(PROCESSED_DIR / f'{FOCUS_GENUS.lower()}_overview.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_growth_form_by_transect(df_gf, form_cols, genus_name=FOCUS_GENUS,
                                    ratio_pair=ratio_pair)
fig.savefig(PROCESSED_DIR / f'{FOCUS_GENUS.lower()}_by_transect.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_growth_form_temporal(df_gf, form_cols, genus_name=FOCUS_GENUS,
                                 month_order=MONTH_ORDER, ratio_pair=ratio_pair)
fig.savefig(PROCESSED_DIR / f'{FOCUS_GENUS.lower()}_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_growth_form_heatmap(df_gf, form_cols, genus_name=FOCUS_GENUS,
                                month_order=MONTH_ORDER)
fig.savefig(PROCESSED_DIR / f'{FOCUS_GENUS.lower()}_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()